# Tutorial 5 – Forced vibration of an N-DoF system
**Vibrations – Centrale Nantes**

Animations of the two forced systems of Tutorial 5 (steady state, small damping $\tilde\zeta_i = 0.02$ on each mode):
* **Animation 1** – double pendulum loaded by a horizontal force on the upper mass (group A),
* **Animation 2** – two masses in series loaded by a force on the first mass (group B).

In each animation, the lower graph gives the steady-state amplitudes versus the forcing frequency $\Omega$; the red line is the current $\Omega$.

**How to use it:** run all the cells (*Run → Run all cells*). In the interactive figures, move the **sliders** to change the parameters and use the **Play/Pause** buttons below the figure.
The code cells are hidden (click on them to show the code).

In [ ]:
#@title Setup (run first): interactive figures with sliders
import sys, subprocess
if "pyodide" in sys.modules:                     # JupyterLite (in the browser)
    import piplite
    await piplite.install(["ipympl", "ipywidgets"])
elif "google.colab" in sys.modules:              # Google Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipympl"])
    from google.colab import output
    output.enable_custom_widget_manager()
try:
    import ipympl
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    print("ipympl is not installed: the animations will be shown without sliders (pip install ipympl).")

In [ ]:
#@title Helper functions (run this cell first)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({"figure.dpi": 90, "animation.embed_limit": 60, "font.size": 10,
                     # LaTeX-like rendering without a LaTeX installation: $...$ formulas and text in STIX (Times-like)
                     "mathtext.fontset": "stix", "font.family": "STIXGeneral"})
BLUE, RED, GREEN, GREY = "#1f4e99", "#a3201d", "#2e6b30", "#777777"


def rk4(f, y0, t):
    """Fixed-step Runge-Kutta 4 integration of dy/dt = f(t, y)."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        h = t[i + 1] - t[i]
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(t[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(t[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y


def spring_xy(p0, p1, n=7, width=0.012, lead=0.15):
    """Zig-zag polyline of a spring between points p0 and p1."""
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    L = np.linalg.norm(p1 - p0)
    u = (p1 - p0) / L
    v = np.array([-u[1], u[0]])
    pts = [p0, p0 + lead * L * u]
    for i in range(2 * n):
        s = lead * L + (1 - 2 * lead) * L * (i + 0.5) / (2 * n)
        pts.append(p0 + s * u + (width if i % 2 == 0 else -width) * v)
    pts += [p1 - lead * L * u, p1]
    pts = np.array(pts)
    return pts[:, 0], pts[:, 1]


def hatch(ax, x0, x1, y, down=True, n=12, size=0.012):
    """Draw a fixed support (line + hatching)."""
    ax.plot([x0, x1], [y, y], color="k", lw=2)
    s = -size if down else size
    for x in np.linspace(x0, x1, n):
        ax.plot([x, x + size], [y, y - s], color="k", lw=0.8)


def make_anim(fig, update, nframes, fps=25):
    """Build an animation and remember what is needed to export it."""
    anim = animation.FuncAnimation(fig, update, frames=nframes, interval=1000 / fps, blit=False)
    anim.update_func, anim.nframes = update, nframes
    return anim


def show(anim):
    """Display an animation in the notebook (works in Jupyter, JupyterLite and Colab)."""
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


def export_frames(anim, name, folder="gifs", dpi=120):
    """Teacher only: save the frames as name-0.png, name-1.png, ... for the LaTeX animate package."""
    import os
    os.makedirs(f"{folder}/{name}", exist_ok=True)
    for i in range(anim.nframes):
        anim.update_func(i)
        anim._fig.savefig(f"{folder}/{name}/{name}-{i}.png", dpi=dpi)
    print(f"{anim.nframes} frames saved in {folder}/{name}/ (last index {anim.nframes - 1})")


# ---------------------------------------------------------------------------
# Live animation: sliders drawn on the figure + Play/Pause widget below it.
# Needs ipympl (Jupyter, JupyterLite, VS Code, Colab). The time steps are driven
# by the ipywidgets "Play" widget (timer in the browser), which also works in
# JupyterLite where matplotlib timers do not run.
# Without ipympl, the animation is shown with the default parameters.
# ---------------------------------------------------------------------------
from matplotlib.widgets import Slider

INTERACTIVE = "ipympl" in plt.get_backend().lower() or "widget" in plt.get_backend().lower()


class LiveAnimation:
    """
    sliders : list of (name, label, vmin, vmax, vinit, step)
    compute : function(params dict) -> data dict (solution on a fine time grid)
    setup   : function(data) -> None, updates the static elements (curves, limits...)
    frame   : function(data, t) -> None, updates the moving elements at time t
    T       : simulated duration (s), played in a loop
              (compute may return data["T"] to change it, e.g. a whole number of periods)
    slow    : slow motion factor
    """

    def __init__(self, fig, sliders, compute, setup, frame, T, slow=1, fps=20):
        self.fig, self.compute, self.setup, self.frame = fig, compute, setup, frame
        self.T, self.slow, self.fps = T, slow, fps
        n = len(sliders)
        fig.subplots_adjust(bottom=0.08 + 0.045 * n)
        self.sliders = {}
        for j, (name, label, vmin, vmax, vinit, step) in enumerate(sliders):
            ax = fig.add_axes([0.22, 0.02 + 0.045 * (n - 1 - j), 0.5, 0.025])
            s = Slider(ax, label, vmin, vmax, valinit=vinit, valstep=step)
            s.on_changed(self._changed)
            self.sliders[name] = s
        for a in fig.axes:                        # drawings with equal aspect: keep them aligned at the top
            if a.get_aspect() == 1.0:
                a.set_anchor("N")
        self.i = 0
        self.data = self.compute(self.params())
        self._loop()
        self.setup(self.data)
        self.frame(self.data, 0.0)
        if INTERACTIVE:
            import ipywidgets as W
            fig.canvas.header_visible = False
            # Play button (timer in the browser) linked to a time slider: the slider can also be moved by hand
            self.play = W.Play(value=0, min=0, max=self.nframes - 1, step=1,
                               interval=int(1000 / fps), repeat=True, description="Play")
            self.tslider = W.IntSlider(value=0, min=0, max=self.nframes - 1, readout=False,
                                       layout=W.Layout(width="45%"))
            W.jslink((self.play, "value"), (self.tslider, "value"))
            self.tlabel = W.Label()
            self.out = W.Output()                 # error messages of the callbacks (if any)
            self.tslider.observe(self._tick, names="value")
            plt.show()                            # figure first, controls below
            display(W.VBox([W.HBox([self.play, self.tslider, self.tlabel]), self.out]))   # paused by default
            self._show_t()
        else:
            print("Static version (default parameters): install ipympl to use the sliders.")
            show(make_anim(fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop), self.nframes, fps))

    def params(self):
        return {name: s.val for name, s in self.sliders.items()}

    def _loop(self):
        """Loop duration (from compute if given) and number of frames of one loop."""
        self.Tloop = self.data.get("T", self.T)
        self.nframes = max(2, int(round(self.Tloop * self.fps * self.slow)))

    def _t(self):
        return (self.i % self.nframes) / self.nframes * self.Tloop

    def _show_t(self):
        self.tlabel.value = f"t = {self._t():.2f} s"

    def _report(self, err):
        import traceback
        with self.out:
            traceback.print_exc()

    def _changed(self, _):
        try:
            self.data = self.compute(self.params())
            self._loop()
            if hasattr(self, "play"):
                self.i = min(self.i, self.nframes - 1)
                self.play.max = self.tslider.max = self.nframes - 1
                self._show_t()
            self.setup(self.data)
            self.frame(self.data, self._t())
            self.fig.canvas.draw_idle()
        except Exception as err:
            self._report(err)

    def _tick(self, change):
        try:
            self.i = change["new"]
            self.frame(self.data, self._t())
            self._show_t()
            self.fig.canvas.draw_idle()
        except Exception as err:
            self._report(err)

    def export(self, name, folder="gifs", dpi=120):
        """Teacher only: frames for the LaTeX animate package (current slider values)."""
        export_frames(make_anim(self.fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop),
                                self.nframes, self.fps), name, folder, dpi)


def period(t, q):
    """Period measured from the upward zero crossings of q(t)."""
    idx = np.where((q[:-1] < 0) & (q[1:] >= 0))[0]
    return np.mean(np.diff(t[idx])) if len(idx) > 1 else np.nan


class Dashpot:
    """Viscous damper drawn between p0 (cylinder side) and p1 (piston side).
    The geometry is fixed by the first call to set() (or after reset()): the piston then moves inside the cylinder."""

    def __init__(self, ax, width=0.01, **kw):
        kw.setdefault("color", "k"); kw.setdefault("lw", 1.2)
        self.w, self.geom = width, None
        self.lines = [ax.plot([], [], **kw)[0] for _ in range(3)]

    def reset(self):
        self.geom = None

    def set(self, p0, p1):
        p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
        L = np.linalg.norm(p1 - p0); u = (p1 - p0) / L; v = np.array([-u[1], u[0]])
        if self.geom is None:
            self.geom = (0.1 * L, 0.65 * L, 0.6 * L)          # stem, cylinder length, piston rod length
        a, cyl, rod = self.geom
        P = lambda s, d=0.0: p0 + s * u + d * v
        stem = np.array([p0, P(a)])
        cylinder = np.array([P(a + cyl, self.w), P(a, self.w), P(a, -self.w), P(a + cyl, -self.w)])
        sp = L - rod
        piston = np.array([P(sp, 0.8 * self.w), P(sp, -0.8 * self.w), P(sp), p1])
        for line, pts in zip(self.lines, [stem, cylinder, piston]):
            line.set_data(pts[:, 0], pts[:, 1])


def double_pendulum_rhs(g, L):
    """Exact equations of a double pendulum with two equal masses and two equal lengths (absolute angles)."""
    def f(_, s):
        t1, t2, w1, w2 = s
        d = t1 - t2
        den = L * (3 - np.cos(2 * d))
        a1 = (-3 * g * np.sin(t1) - g * np.sin(t1 - 2 * t2) - 2 * np.sin(d) * (w2 ** 2 * L + w1 ** 2 * L * np.cos(d))) / den
        a2 = (2 * np.sin(d) * (2 * w1 ** 2 * L + 2 * g * np.cos(t1) + w2 ** 2 * L * np.cos(d))) / den
        return np.array([w1, w2, a1, a2])
    return f


def draw_two_masses(ax, x1, x2, objs=None, gap=0.3, w=0.12, h=0.1, scale=1.0):
    """Two masses between two walls (walls at 0 and 3*gap+2*w); x1, x2 in the units of the drawing."""
    xa = gap + w / 2 + scale * x1
    xb = 2 * gap + 1.5 * w + scale * x2
    xr = 3 * gap + 2 * w
    if objs is None:
        hatch_v = lambda x, s: [ax.plot([x, x], [-0.02, h + 0.06], "k", lw=2)] + \
            [ax.plot([x, x + s], [y, y + 0.02], "k", lw=0.7) for y in np.linspace(-0.02, h + 0.04, 7)]
        hatch_v(0, -0.025); hatch_v(xr, 0.025)
        ax.plot([0, xr], [-0.02, -0.02], "k", lw=1)
        sps = [ax.plot([], [], "k", lw=1.1)[0] for _ in range(3)]
        boxes = [plt.Rectangle((0, 0), w, h, fc="#f4c7c3", ec="k") for _ in range(2)]
        for b in boxes:
            ax.add_patch(b)
        objs = (sps, boxes)
    sps, boxes = objs
    yc = h / 2
    ends = [(0, xa - w / 2), (xa + w / 2, xb - w / 2), (xb + w / 2, xr)]
    for s, (p, q) in zip(sps, ends):
        s.set_data(*spring_xy((p, yc), (q, yc), n=6, width=0.025))
    boxes[0].set_x(xa - w / 2); boxes[1].set_x(xb - w / 2)
    return objs


## Animation 1 – Forced double pendulum (group A)
**Observe:** the amplitudes of the two masses when $\Omega$ varies. For which values is the response very large? For which value does the **upper** mass (the loaded one) stop moving, and what does the lower mass do then?

In [ ]:
#@title Animation 1
def animation_1():
    m, g, L, z = 0.2, 9.81, 0.25, 0.02
    M = m * L ** 2 * np.array([[2, 1], [1, 1]]); K = m * g * L * np.array([[2, 0], [0, 1]])
    Phi = np.array([[1, 1], [np.sqrt(2), -np.sqrt(2)]])
    mt, kt = np.diag(Phi.T @ M @ Phi), np.diag(Phi.T @ K @ Phi); wi = np.sqrt(kt / mt)

    def X_of(W, F0):
        Qt = Phi.T @ np.array([F0 * L, 0.0])                                       # forces on the decoupled equations
        Xt = Qt / (kt - mt * W ** 2 + 2j * z * mt * wi * W)
        return Phi @ Xt                                                            # complex amplitudes of theta_1, theta_2

    Ws = np.linspace(0.5, 15, 1500)
    def compute(p):
        W, F0 = p["W"], p["F0"] / 1000
        X = X_of(W, F0); A = np.abs(np.array([X_of(w, F0) for w in Ws]))
        return dict(W=W, X=X, A=A, T=2 * np.pi / W)

    fig = plt.figure(figsize=(9.5, 6.8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.8], height_ratios=[1, 1], top=0.94, wspace=0.25, hspace=0.35)
    ax = fig.add_subplot(gs[:, 0]); axf = fig.add_subplot(gs[:, 1])
    ax.set_aspect("equal"); ax.axis("off"); R = 2.1 * L
    ax.set_xlim(-0.75 * R, 0.75 * R); ax.set_ylim(-R, 0.1 * R)
    hatch(ax, -0.15 * R, 0.15 * R, 0.0, down=False, size=0.03 * R)
    ax.plot([0, 0], [0, -2 * L], color=GREY, ls="--", lw=0.8)
    rods, = ax.plot([], [], "k", lw=1.5)
    b1, = ax.plot([], [], "o", color=BLUE, ms=11); b2, = ax.plot([], [], "o", color=RED, ms=11)
    arrow = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2))
    ax.set_title("forced double pendulum")
    f1, = axf.plot([], [], color=BLUE, lw=2, label=r"$|\theta_1|$ (loaded)")
    f2, = axf.plot([], [], color=RED, lw=2, ls="--", label=r"$|\theta_2|$")
    vline = axf.axvline(0, color=RED, lw=1)
    axf.set_xlim(Ws[0], Ws[-1]); axf.set_yscale("log"); axf.set_ylim(1e-3, 100)
    axf.set_xlabel(r"$\Omega$ (rad/s)"); axf.set_ylabel("steady-state amplitude (°)"); axf.grid(alpha=0.3, which="both"); axf.legend(loc="upper right")
    info = axf.set_title("")

    def setup(d):
        f1.set_data(Ws, np.degrees(d["A"][:, 0])); f2.set_data(Ws, np.degrees(d["A"][:, 1]))
        vline.set_xdata([d["W"]] * 2)
        info.set_text(rf"$|\theta_1|$ = {np.degrees(abs(d['X'][0])):.2f}°   $|\theta_2|$ = {np.degrees(abs(d['X'][1])):.2f}°")

    def frame(d, t):
        th = np.real(d["X"] * np.exp(1j * d["W"] * t))
        th = np.clip(th, -1.2, 1.2)                                                # drawing only
        x1, y1 = L * np.sin(th[0]), -L * np.cos(th[0]); x2, y2 = x1 + L * np.sin(th[1]), y1 - L * np.cos(th[1])
        rods.set_data([0, x1, x2], [0, y1, y2]); b1.set_data([x1], [y1]); b2.set_data([x2], [y2])
        f = np.cos(d["W"] * t)
        arrow.set_position((x1 - 0.18, y1)); arrow.xy = (x1 - 0.18 + 0.12 * f, y1)

    live = LiveAnimation(fig, [("W", r"$\Omega$ (rad/s)", 1, 15, 4.79, 0.01), ("F0", r"$F_0$ (mN)", 5, 100, 50, 5)],
                         compute, setup, frame, 4.0, slow=1)
    return live

live1 = animation_1()

## Animation 2 – Forced masses in series (group B)
**Observe:** the amplitudes of the two masses when $\Omega$ varies. For which values is the response very large? For which value does the **first** mass (the loaded one) stop moving, and what does the second mass do then?

In [ ]:
#@title Animation 2
def animation_2():
    m, k, z = 0.5, 200.0, 0.02
    M = m * np.eye(2); K = k * np.array([[2, -1], [-1, 2]])
    Phi = np.array([[1, 1], [1, -1]])
    mt, kt = np.diag(Phi.T @ M @ Phi), np.diag(Phi.T @ K @ Phi); wi = np.sqrt(kt / mt)

    def X_of(W, F0):
        Xt = (Phi.T @ np.array([F0, 0.0])) / (kt - mt * W ** 2 + 2j * z * mt * wi * W)
        return Phi @ Xt

    Ws = np.linspace(1, 50, 1500)
    def compute(p):
        W, F0 = p["W"], p["F0"]
        return dict(W=W, X=X_of(W, F0), A=np.abs(np.array([X_of(w, F0) for w in Ws])))

    fig = plt.figure(figsize=(9.5, 6.8))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.6], top=0.94, hspace=0.3)
    ax, axf = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    ax.set_aspect("equal"); ax.axis("off"); ax.set_xlim(-0.05, 1.2); ax.set_ylim(-0.05, 0.2)
    objs = draw_two_masses(ax, 0, 0)
    arrow = ax.annotate("", xy=(0, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2))
    f1, = axf.plot([], [], color=BLUE, lw=2, label="$|x_1|$ (loaded)")
    f2, = axf.plot([], [], color=RED, lw=2, ls="--", label="$|x_2|$")
    vline = axf.axvline(0, color=RED, lw=1)
    axf.set_xlim(Ws[0], Ws[-1]); axf.set_yscale("log"); axf.set_ylim(1e-3, 100)
    axf.set_xlabel(r"$\Omega$ (rad/s)"); axf.set_ylabel("steady-state amplitude (cm)"); axf.grid(alpha=0.3, which="both"); axf.legend(loc="upper right")
    info = axf.set_title("")

    def setup(d):
        f1.set_data(Ws, 100 * d["A"][:, 0]); f2.set_data(Ws, 100 * d["A"][:, 1])
        vline.set_xdata([d["W"]] * 2)
        info.set_text(rf"$|x_1|$ = {100 * abs(d['X'][0]):.2f} cm   $|x_2|$ = {100 * abs(d['X'][1]):.2f} cm")

    def frame(d, t):
        x = 100 * np.real(d["X"] * np.exp(1j * d["W"] * t))
        x = np.clip(x, -6, 6)                                                      # drawing only
        draw_two_masses(ax, x[0], x[1], objs, scale=0.025)
        xa = 0.3 + 0.06 + 0.025 * x[0]; f = np.cos(d["W"] * t)
        arrow.set_position((xa, 0.15)); arrow.xy = (xa + 0.08 * f, 0.15)

    live = LiveAnimation(fig, [("W", r"$\Omega$ (rad/s)", 1, 50, 20, 0.1), ("F0", r"$F_0$ (N)", 0.1, 2, 0.5, 0.1)],
                         compute, setup, frame, 2.0, slow=2)
    fig.text(0.02, 0.005, "(drawing: displacements ×2.5)", fontsize=7, color=GREY)
    return live

live2 = animation_2()

---
*Teacher only – export the frames of an animation for the LaTeX `animate` package* (uncomment and run):

In [ ]:
# live1.export("td5DoublePendulumForced")
# live2.export("td5TwoMassesForced")